# 04 — Silver Master Data

This notebook cleans the three master tables from Bronze:
- Customer
- Merchant
- Card

For each table: **Read Bronze → Cast columns → Check bad rows and duplicates → Save Silver and Quarantine**.

The Bronze tables are not changed.

In [0]:
from pyspark.sql.functions import col, trim, upper, when, lit

# Database names
BRONZE = "capstone_team01.bronze_schema"
SILVER = "capstone_team01.silver_schema"

# Create Silver schema
spark.sql("CREATE SCHEMA IF NOT EXISTS capstone_team01.silver_schema")

## 1. Customer master

In [0]:
# Read Bronze table
customer = spark.table(f"{BRONZE}.bronze_customer_master")

# Cast columns
customer = (customer
    .withColumn("customer_id", trim(col("customer_id").cast("string")))
    .withColumn("customer_segment", trim(col("customer_segment").cast("string")))
    .withColumn("onboarded_date", col("onboarded_date").cast("date"))
    .withColumn("home_country", trim(col("home_country").cast("string")))
    .withColumn("account_status", upper(trim(col("account_status").cast("string"))))
    .withColumn("kyc_status", upper(trim(col("kyc_status").cast("string"))))
    .withColumn("risk_band", upper(trim(col("risk_band").cast("string"))))
    .withColumn("relationship_manager", trim(col("relationship_manager").cast("string")))
    .withColumn("city", trim(col("city").cast("string")))
    .withColumn("customer_type", trim(col("customer_type").cast("string")))
    .drop("_rescued_data")
)

customer.printSchema()
display(customer.limit(5))

In [0]:
# Separate records with missing required values
customer_invalid = customer.filter(
    col("customer_id").isNull() |
    (col("customer_id") == "") |
    col("onboarded_date").isNull()
)

# Keep records with required values
customer_valid = customer.filter(
    col("customer_id").isNotNull() &
    (col("customer_id") != "") &
    col("onboarded_date").isNotNull()
)

# Keep one row per customer_id
customer_silver = customer_valid.dropDuplicates(["customer_id"])

# Get the extra duplicate rows
customer_duplicates = customer_valid.exceptAll(customer_silver)

# Add reasons to quarantine rows
customer_invalid = customer_invalid.withColumn("dq_reason", lit("Missing customer_id or onboarded_date"))
customer_duplicates = customer_duplicates.withColumn("dq_reason", lit("Duplicate customer_id"))

# Combine invalid and duplicate rows
customer_quarantine = customer_invalid.unionByName(customer_duplicates)

# Save tables
customer_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_customer_master")
customer_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.quarantine_customer_master")

print("Bronze:", customer.count())
print("Silver:", customer_silver.count())
print("Quarantine:", customer_quarantine.count())

## 2. Merchant master

In [0]:
# Read Bronze table
merchant = spark.table(f"{BRONZE}.bronze_merchant_master")

# Cast columns
merchant = (merchant
    .withColumn("merchant_id", trim(col("merchant_id").cast("string")))
    .withColumn("merchant_name", trim(col("merchant_name").cast("string")))
    .withColumn("merchant_category", trim(col("merchant_category").cast("string")))
    .withColumn("merchant_country", trim(col("merchant_country").cast("string")))
    .withColumn("onboarded_date", col("onboarded_date").cast("date"))
    .withColumn("merchant_status", upper(trim(col("merchant_status").cast("string"))))
    .withColumn("settlement_currency", upper(trim(col("settlement_currency").cast("string"))))
    .withColumn("risk_band", upper(trim(col("risk_band").cast("string"))))
    .withColumn("city", trim(col("city").cast("string")))
    .withColumn("acquirer", trim(col("acquirer").cast("string")))
    .drop("_rescued_data")
)

merchant.printSchema()
display(merchant.limit(5))

In [0]:
# Separate records with missing required values
merchant_invalid = merchant.filter(
    col("merchant_id").isNull() |
    (col("merchant_id") == "") |
    col("merchant_name").isNull() |
    (col("merchant_name") == "") |
    col("onboarded_date").isNull()
)

# Keep records with required values
merchant_valid = merchant.filter(
    col("merchant_id").isNotNull() &
    (col("merchant_id") != "") &
    col("merchant_name").isNotNull() &
    (col("merchant_name") != "") &
    col("onboarded_date").isNotNull()
)

# Keep one row per merchant_id
merchant_silver = merchant_valid.dropDuplicates(["merchant_id"])

# Get the extra duplicate rows
merchant_duplicates = merchant_valid.exceptAll(merchant_silver)

# Add reasons to quarantine rows
merchant_invalid = merchant_invalid.withColumn("dq_reason", lit("Missing merchant_id, merchant_name or onboarded_date"))
merchant_duplicates = merchant_duplicates.withColumn("dq_reason", lit("Duplicate merchant_id"))

# Combine invalid and duplicate rows
merchant_quarantine = merchant_invalid.unionByName(merchant_duplicates)

# Save tables
merchant_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_merchant_master")
merchant_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.quarantine_merchant_master")

print("Bronze:", merchant.count())
print("Silver:", merchant_silver.count())
print("Quarantine:", merchant_quarantine.count())

## 3. Card master

In [0]:
# Read Bronze table
card = spark.table(f"{BRONZE}.bronze_card_master")

# Cast columns
card = (card
    .withColumn("card_id", trim(col("card_id").cast("string")))
    .withColumn("customer_id", trim(col("customer_id").cast("string")))
    .withColumn("card_type", trim(col("card_type").cast("string")))
    .withColumn("issuer_country", trim(col("issuer_country").cast("string")))
    .withColumn("issue_date", col("issue_date").cast("date"))
    .withColumn("expiry_date", col("expiry_date").cast("date"))
    .withColumn("card_status", upper(trim(col("card_status").cast("string"))))
    .withColumn("network", trim(col("network").cast("string")))
    .withColumn("daily_limit", col("daily_limit").cast("decimal(18,2)"))
    .withColumn(
        "tokenized_flag",
        when(upper(trim(col("tokenized_flag").cast("string"))).isin("Y", "YES", "TRUE", "1"), True)
        .when(upper(trim(col("tokenized_flag").cast("string"))).isin("N", "NO", "FALSE", "0"), False)
        .otherwise(None)
    )
)

card.printSchema()
display(card.limit(5))

In [0]:
# Check customer IDs against Silver customer master
customer_ids = spark.table(f"{SILVER}.silver_customer_master").select("customer_id").dropDuplicates()

card = card.join(customer_ids.withColumn("customer_exists", lit("yes")), "customer_id", "left")

# Separate invalid records
card_invalid = card.filter(
    col("card_id").isNull() |
    (col("card_id") == "") |
    col("customer_id").isNull() |
    col("customer_exists").isNull() |
    col("issue_date").isNull() |
    col("expiry_date").isNull() |
    (col("expiry_date") < col("issue_date")) |
    col("daily_limit").isNull() |
    col("tokenized_flag").isNull()
)

# Keep valid records
card_valid = card.filter(
    col("card_id").isNotNull() &
    (col("card_id") != "") &
    col("customer_id").isNotNull() &
    col("customer_exists").isNotNull() &
    col("issue_date").isNotNull() &
    col("expiry_date").isNotNull() &
    (col("expiry_date") >= col("issue_date")) &
    col("daily_limit").isNotNull() &
    col("tokenized_flag").isNotNull()
)

# Remove temporary column
card_invalid = card_invalid.drop("customer_exists")
card_valid = card_valid.drop("customer_exists")

# Keep one row per card_id
card_silver = card_valid.dropDuplicates(["card_id"])

# Get the extra duplicate rows
card_duplicates = card_valid.exceptAll(card_silver)

# Add reasons to quarantine rows
card_invalid = card_invalid.withColumn("dq_reason", lit("Invalid card data or customer reference"))
card_duplicates = card_duplicates.withColumn("dq_reason", lit("Duplicate card_id"))

# Combine invalid and duplicate rows
card_quarantine = card_invalid.unionByName(card_duplicates)

# Save tables
card_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.silver_card_master")
card_quarantine.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{SILVER}.quarantine_card_master")

print("Bronze:", card.count())
print("Silver:", card_silver.count())
print("Quarantine:", card_quarantine.count())

## 4. Final checks

In [0]:
# Check Silver and Quarantine counts
print("Customer Silver:", spark.table(f"{SILVER}.silver_customer_master").count())
print("Customer Quarantine:", spark.table(f"{SILVER}.quarantine_customer_master").count())

print("Merchant Silver:", spark.table(f"{SILVER}.silver_merchant_master").count())
print("Merchant Quarantine:", spark.table(f"{SILVER}.quarantine_merchant_master").count())

print("Card Silver:", spark.table(f"{SILVER}.silver_card_master").count())
print("Card Quarantine:", spark.table(f"{SILVER}.quarantine_card_master").count())

# Check duplicate IDs in Silver
display(spark.table(f"{SILVER}.silver_customer_master").groupBy("customer_id").count().filter(col("count") > 1))
display(spark.table(f"{SILVER}.silver_merchant_master").groupBy("merchant_id").count().filter(col("count") > 1))
display(spark.table(f"{SILVER}.silver_card_master").groupBy("card_id").count().filter(col("count") > 1))

In [0]:
from pyspark.sql.functions import col

card_invalid = spark.table("capstone_team01.silver_schema.quarantine_card_master").filter(col("dq_reason") == "Invalid card data or customer reference")

# Find cards with invalid daily limits
invalid_limits = card_invalid.filter(
    col("daily_limit").isNull() |
    (col("daily_limit") <= 0)
)

invalid_count = invalid_limits.count()

if invalid_count == 0:
    print("Data is clean: no null or invalid (<= 0) daily_limit values found in card master.")
else:
    print(f"Found {invalid_count} cards with null or invalid values:")
    display(invalid_limits)